# Setup

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from pathlib import Path
from copy import deepcopy
import multiprocessing as mp
import logging
import random

import mlflow
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import numpy as np
from dotenv import load_dotenv

from halcyon.Simulators.SimulationResult import SimulationResult
from halcyon.results.manipulate_results_df import (create_tidy_df_from_mlflow_runs, 
                                                   split_df_into_single_and_double_ko,
                                                   add_model_name_to_df,
                                                   df_with_ground_truth_as_kind_of_model)

from halcyon.results.plot_from_df import plot_model_plots, plot_per_gene_mse_heatmap, rotate_x_tick_labels, wrap_titles

from halcyon.results import DfColumnNames

seed = 174
# for CPU
torch.manual_seed(seed)              
# for all GPUs
torch.cuda.manual_seed_all(seed)     
np.random.seed(seed) 
random.seed(seed) 

load_dotenv() 

logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s - %(levelname)s - %(message)s',
    force=True  # Important in Jupyter to reset config
)

# Load from MLFlow

In [2]:
runs_to_download = []

## Best ML runs

In [3]:
filter_string = "params.lambda_l1 = '0.0' and params.hidden_size = '10' and params.nr_layers = '2' and params.synthesis_only_positive = 'False'"
runs_to_download.extend(mlflow.search_runs(filter_string=filter_string, experiment_ids=['4'], output_format='list'))

In [4]:
len(runs_to_download)

3

## Hybrid and MM runs

In [5]:
hybrid_and_mm_runs = mlflow.search_runs(filter_string="attributes.status = 'FINISHED'", experiment_ids=['6'], output_format='list')

In [6]:
len(hybrid_and_mm_runs)
runs_to_download.extend(hybrid_and_mm_runs)

In [7]:
len(runs_to_download)

38

In [8]:
raw_predictions_df = create_tidy_df_from_mlflow_runs(runs_to_download)

2026-04-10 14:29:08,116 - INFO - Downloading artifacts for run e51261c9ab8644edaef773a45d9d8551


2026-04-10 14:29:09,406 - INFO - Downloading artifacts for run 0d56cdfb0da342c5abfc20196377c1fd


2026-04-10 14:29:10,434 - INFO - Downloading artifacts for run 6a3375f5c070446f996b2719f487da25


2026-04-10 14:29:11,429 - INFO - Downloading artifacts for run 1914c2fd8fd949fea18c15a6c7ed7073


2026-04-10 14:29:12,588 - INFO - Downloading artifacts for run 973239ca0ebb4fd182b2c97f6f17b195


...

2026-04-10 14:29:42,199 - INFO - Downloading artifacts for run 179aaca376b248a7a102499c150dc642


2026-04-10 14:29:43,091 - INFO - Downloading artifacts for run 0b0fa46aa0ec43d2a18e78ec8bfe386e


2026-04-10 14:29:43,986 - INFO - Downloading artifacts for run 8736d8075cf34be390d737bf3740c151


2026-04-10 14:29:44,894 - INFO - Downloading artifacts for run 5eab6075d6144f08bd03d7b44aea6a80


2026-04-10 14:29:45,799 - INFO - Downloading artifacts for run d4adee2c979c4452b9e970152af0b3f4


In [10]:
raw_predictions_df['condition'].unique()

array(['KO: Node_52. Env: None', 'KO: Node_78. Env: None',
       'KO: Node_71. Env: None', 'KO: Node_19. Env: None',
       'KO: Node_45. Env: None', 'KO: Node_3. Env: None',
       'KO: Node_51. Env: None', 'KO: Node_27. Env: None',
       'KO: Node_97. Env: None', 'KO: Node_25. Env: None',
       'KO: Node_1. Env: None', 'KO: Node_16. Env: None',
       'KO: Node_99. Env: None', 'KO: Node_55. Env: None',
       'KO: Node_79. Env: None', 'KO: Node_46. Env: None',
       'KO: Node_93. Env: None', 'KO: Node_60. Env: None',
       'KO: Node_22. Env: None', 'KO: Node_83. Env: None',
       'KO: Node_59. Env: None', 'KO: Node_24. Env: None',
       'KO: Node_74. Env: None', 'KO: Node_21. Env: None',
       'KO: Node_90. Env: None', 'KO: Node_69. Env: None',
       'KO: Node_17. Env: None', 'KO: Node_43. Env: None',
       'KO: Node_35. Env: None', 'KO: Node_40. Env: None',
       'KO: Node_13. Env: None', 'KO: Node_48. Env: None',
       'KO: Node_26. Env: None', 'KO: Node_12. Env: None',

In [9]:
raw_predictions_df.head()

,gene,time,predicted_expression,ground_truth,condition,run_id,parent_run_id
0,Node_0,0.0,0.417963,0.417963,KO: Node_52. Env: None,e51261c9ab8644edaef773a45d9d8551,e56b1774f68349209f3b7e672158fa01
1,Node_1,0.0,0.393894,0.393894,KO: Node_52. Env: None,e51261c9ab8644edaef773a45d9d8551,e56b1774f68349209f3b7e672158fa01
2,Node_2,0.0,0.071135,0.071135,KO: Node_52. Env: None,e51261c9ab8644edaef773a45d9d8551,e56b1774f68349209f3b7e672158fa01
3,Node_3,0.0,0.455552,0.455552,KO: Node_52. Env: None,e51261c9ab8644edaef773a45d9d8551,e56b1774f68349209f3b7e672158fa01
4,Node_4,0.0,0.562109,0.562109,KO: Node_52. Env: None,e51261c9ab8644edaef773a45d9d8551,e56b1774f68349209f3b7e672158fa01


Add number of genes

In [11]:
records = []
for run_id in raw_predictions_df['parent_run_id'].unique():
    records.append({
        'parent_run_id': run_id,
        'nr_genes': mlflow.get_run(run_id).data.params['nr_genes']
    })
nr_genes_df = pd.DataFrame(records)
nr_genes_df.head()

,parent_run_id,nr_genes
0,e56b1774f68349209f3b7e672158fa01,100
1,19ba6fc0e8094a66a68f2f67cdca2162,50
2,d660c1d337c34d98a349605d9b4d5b47,20


Add noiseless expression values

In [ ]:
records = []
for run_id in raw_predictions_df['parent_run_id'].unique():
    ml_flow_download_path = mlflow.artifacts.download_artifacts(
        run_id=run_id, artifact_path="dataset"
    )
    sim_output = SimulationResult.load_from_pkl(
        Path(ml_flow_download_path) / "simulation_result.pkl"
    )
    long_form_simulation = sim_output.to_long_form_df()
    long_form_simulation[DfColumnNames.RUN_ID_COL] = run_id
    records.append(long_form_simulation)
long_form_simulation = pd.concat(records, ignore_index=True)

2026-04-10 14:29:53,377 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-04-10 14:29:53,415 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-04-10 14:29:53,436 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-04-10 14:29:53,442 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-04-10 14:29:53,461 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-04-10 14:29:53,466 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-04-10 14:29:53,509 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-04-10 14:29:53,518 - WARNING - Connection p

2026-04-10 14:29:59,171 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-04-10 14:29:59,213 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-04-10 14:29:59,217 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-04-10 14:29:59,223 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-04-10 14:29:59,242 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-04-10 14:29:59,255 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-04-10 14:29:59,272 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-04-10 14:29:59,308 - WARNING - Connection p

2026-04-10 14:30:04,641 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-04-10 14:30:04,660 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-04-10 14:30:04,679 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-04-10 14:30:04,695 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-04-10 14:30:04,715 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-04-10 14:30:04,746 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-04-10 14:30:04,750 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-04-10 14:30:04,774 - WARNING - Connection p

In [15]:
raw_predictions_df.head()

,gene,time,predicted_expression,ground_truth,condition,run_id,parent_run_id
0,Node_0,0.0,0.417963,0.417963,KO: Node_52. Env: None,e51261c9ab8644edaef773a45d9d8551,e56b1774f68349209f3b7e672158fa01
1,Node_1,0.0,0.393894,0.393894,KO: Node_52. Env: None,e51261c9ab8644edaef773a45d9d8551,e56b1774f68349209f3b7e672158fa01
2,Node_2,0.0,0.071135,0.071135,KO: Node_52. Env: None,e51261c9ab8644edaef773a45d9d8551,e56b1774f68349209f3b7e672158fa01
3,Node_3,0.0,0.455552,0.455552,KO: Node_52. Env: None,e51261c9ab8644edaef773a45d9d8551,e56b1774f68349209f3b7e672158fa01
4,Node_4,0.0,0.562109,0.562109,KO: Node_52. Env: None,e51261c9ab8644edaef773a45d9d8551,e56b1774f68349209f3b7e672158fa01


In [14]:
long_form_simulation.head()

,gene,time,ground_truth,condition,run_id
0,Node_0,0.0,0.513996,KO: Node_0. Env: None,e56b1774f68349209f3b7e672158fa01
1,Node_1,0.0,0.428341,KO: Node_0. Env: None,e56b1774f68349209f3b7e672158fa01
2,Node_2,0.0,0.156379,KO: Node_0. Env: None,e56b1774f68349209f3b7e672158fa01
3,Node_3,0.0,0.413022,KO: Node_0. Env: None,e56b1774f68349209f3b7e672158fa01
4,Node_4,0.0,0.414592,KO: Node_0. Env: None,e56b1774f68349209f3b7e672158fa01


Merging

In [24]:
outcome = pd.merge(
    raw_predictions_df, 
    long_form_simulation, 
    left_on=['parent_run_id', 'gene', 'time', 'condition'], 
    right_on=['run_id', 'gene', 'time', 'condition'], 
    validate='m:1',
    suffixes=['_from_pred_df', '_from_simulation'])

In [25]:
np.isclose(outcome['ground_truth_from_pred_df'], outcome['ground_truth_from_simulation']).all()

np.True_

In [18]:
outcome.head()

,gene,time,predicted_expression,ground_truth_x,condition,run_id_x,parent_run_id,ground_truth_y,run_id_y
0,Node_0,0.0,0.417963,0.417963,KO: Node_52. Env: None,e51261c9ab8644edaef773a45d9d8551,e56b1774f68349209f3b7e672158fa01,0.417963,e56b1774f68349209f3b7e672158fa01
1,Node_1,0.0,0.393894,0.393894,KO: Node_52. Env: None,e51261c9ab8644edaef773a45d9d8551,e56b1774f68349209f3b7e672158fa01,0.393894,e56b1774f68349209f3b7e672158fa01
2,Node_2,0.0,0.071135,0.071135,KO: Node_52. Env: None,e51261c9ab8644edaef773a45d9d8551,e56b1774f68349209f3b7e672158fa01,0.071135,e56b1774f68349209f3b7e672158fa01
3,Node_3,0.0,0.455552,0.455552,KO: Node_52. Env: None,e51261c9ab8644edaef773a45d9d8551,e56b1774f68349209f3b7e672158fa01,0.455552,e56b1774f68349209f3b7e672158fa01
4,Node_4,0.0,0.562109,0.562109,KO: Node_52. Env: None,e51261c9ab8644edaef773a45d9d8551,e56b1774f68349209f3b7e672158fa01,0.562109,e56b1774f68349209f3b7e672158fa01


In [ ]:


with_model_name = add_model_name_to_df(raw_predictions_df)

## Make subsets of data for plotting

In [ ]:
# Split into Single and Double knockout
tidy_df_single_ko, tidy_df_double_ko = split_df_into_single_and_double_ko(with_model_name)
tidy_df_single_ko['condition'].value_counts()

In [ ]:
tidy_df_single_ko[DfColumnNames.MODEL_COL].value_counts()

In [ ]:
def df_selection_function(df):
    node_index_regex = '|'.join([f'{i}' for i in range(4)])
    genes_to_show_regex = rf"Node_(?:{node_index_regex})(?:[.,]|$)"

    models_to_show = ["PhoenixAddAndProdLayers", "PyTorchODE", DfColumnNames.GROUND_TRUTH_COL, "HybridModel"]
    models_to_show_regex = "|".join(models_to_show)

    index_to_select = ((df[DfColumnNames.GENE_COL].str.contains(genes_to_show_regex))
            & (df[DfColumnNames.CONDITION_COL].str.contains(genes_to_show_regex))
            & (df[DfColumnNames.MODEL_COL].str.contains(models_to_show_regex)))
    filtered_df =  df[index_to_select]
    assert len(filtered_df), 'Filtered dataframe is empty'
    return filtered_df

In [ ]:
df_selection_function(tidy_df_single_ko).sum()

In [ ]:
filtered_df_single_ko = df_selection_function(tidy_df_single_ko)
filtered_df_double_ko = df_selection_function(tidy_df_double_ko)

# Plotting

## Lineplots

### Single KO

In [ ]:
# FOr each of this knockout, plot predicted vs real for all model (for subset of genes)

# Plot for single KO
g = plot_model_plots(filtered_df_single_ko)
wrap_titles(g)

### Double KO

In [ ]:
g = plot_model_plots(filtered_df_double_ko)
wrap_titles(g)

## Now the heatmaps

### Single KO

In [ ]:
from matplotlib.colors import Normalize

In [ ]:
# Calculate the MSE heatmap that we've shown before
g = plot_per_gene_mse_heatmap(tidy_df_single_ko, robust_cmap=False)
wrap_titles(g)

### Double KO

In [ ]:
# Calculate the MSE heatmap that we've shown before
g = plot_per_gene_mse_heatmap(tidy_df_double_ko, robust_cmap=False)
wrap_titles(g)